# Create New Angle Types Based on the Data

In [ ]:
import concurrent.futures
import io
import json
import math
import os
from abc import ABC, abstractmethod
from collections import Counter, defaultdict, namedtuple
from concurrent.futures import ProcessPoolExecutor, as_completed
from copy import deepcopy
from functools import partial
from pathlib import Path
from typing import Callable, Iterable, Optional

import numpy as np
from datasets import Dataset
from openff.interchange import Interchange
from openff.toolkit import ForceField, Molecule
from openff.toolkit.topology import Topology
from openff.toolkit.typing.engines.smirnoff.parameters import (
    AngleHandler,
    BondHandler,
    ImproperTorsionHandler,
    ParameterHandler,
    ParameterList,
    ParameterType,
    ProperTorsionHandler,
)
from openff.units import unit as off_unit
from PIL import Image
from rdkit import Chem
from rdkit.Chem import Draw, rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
from tqdm import tqdm
from yammbs.torsion.inputs import QCArchiveTorsionDataset

_DEGREES = off_unit.degrees
_RADIANS = off_unit.radians
_KCAL_PER_MOL = off_unit.kilocalories_per_mole

SMILES_JSON_PATH = "../data/spice2/smiles_test_train.json"

with open(SMILES_JSON_PATH, "r") as f:
    smiles_data = json.load(f)

# Also get the SMILES after filtering to make sure all types are trained
SMILES_TRAIN_FILTERED_DATASET = Dataset.load_from_disk(
    "../data/spice2/data-filtered-lj-sage-2-2-msm-0-expanded-torsions/data-train"
)
SMILES_TRAIN_FILTERED = list(SMILES_TRAIN_FILTERED_DATASET["smiles"])

TNETDATA = Path(
    "../benchmarking/torsion_benchmarks/torsionnet_500/input_data/qca-torsion-data.json"
)
BIARYL_DATA = Path(
    "../benchmarking/torsion_benchmarks/rowley_biaryl/input_data/qca-torsion-data.json"
)


def get_torsion_smiles(data_path: Path) -> set[str]:
    torsion_data = QCArchiveTorsionDataset.model_validate_json(data_path.read_text())
    return set([t.mapped_smiles for t in torsion_data.qm_torsions])


TNET_SMILES = get_torsion_smiles(TNETDATA)
BIARYL_SMILES = get_torsion_smiles(BIARYL_DATA)

INDUSTRY_BENCHMARK_DATA = Path(
    "../benchmarking/industry_benchmark/input_data/filtered-industry-cached.json"
)


def get_qcdataset_smiles(data_path: Path) -> set[str]:
    dataset = json.loads(data_path.read_text())
    return set([entry["mapped_smiles"] for entry in dataset])


INDUSTRY_BENCHMARK_SMILES = get_qcdataset_smiles(INDUSTRY_BENCHMARK_DATA)

# Get the linear torsions
linear_torsions_ff = ForceField("linear_torsions.offxml")
linear_torsion_parameters = linear_torsions_ff.get_parameter_handler(
    "ProperTorsions"
).parameters
linear_torsion_smirks = [param.smirks for param in linear_torsion_parameters]

In [2]:
GetAtomSmirksFn = Callable[[int, int, Chem.Mol], str]
GetBondsSmirksFn = Callable[[tuple[int, int], bool, Chem.Mol], str]
SpecificityLevel = namedtuple(
    "SpecificityLevel", ["name", "get_atom_smirks", "get_bond_smirks"]
)


def get_atom_descriptors(at_idx: int, mol: Chem.Mol) -> dict[str, str]:
    """Generate SMRIKS-ready descriptors for an atom in the molecule."""
    descriptors = {
        "atomic_num": f"#{mol.GetAtomWithIdx(at_idx).GetAtomicNum()}",
        "degree": f"X{mol.GetAtomWithIdx(at_idx).GetDegree()}",
        "charge": mol.GetAtomWithIdx(at_idx).GetFormalCharge(),
    }
    descriptors["charge"] = (
        f"+{descriptors['charge']}"
        if descriptors["charge"] >= 0
        else str(descriptors["charge"])
    )

    return descriptors


def get_bond_type(atom_idxs: tuple[int, int], mol: Chem.Mol) -> str:
    """Get the bond type between two atoms in the molecule.
    Returns a string representation of the bond type.
    """
    bond = mol.GetBondBetweenAtoms(*atom_idxs)
    if bond is None:
        raise ValueError(f"No bond found between atoms {atom_idxs} in the molecule.")

    bt = bond.GetBondType()
    if bt == Chem.BondType.SINGLE:
        return "-"
    elif bt == Chem.BondType.DOUBLE:
        return "="
    elif bt == Chem.BondType.TRIPLE:
        return "#"
    elif bt == Chem.BondType.AROMATIC:
        return ":"
    else:
        return "~"


def get_atom_smirks_standard(
    at_idx: int, at_id: int, mol: Chem.Mol, terminal_idxs: tuple[int, int]
) -> str:
    """Generate the SMIRKS representation for an atom in the torsion.
    Standard specificity level.
    """
    ds = get_atom_descriptors(at_idx, mol)
    return f"[{ds['atomic_num']}{ds['degree']}:{at_id + 1}]"


def _get_atom_smirks_all_bonded(
    at_idx: int,
    at_id: int,
    mol: Chem.Mol,
    central_only: bool,
    terminal_idxs: tuple[int, int],
) -> str:
    """Get a very specific representation the central atoms in a torsion which specifies all
    neighbouring atoms and their bond types.
    """
    ds = get_atom_descriptors(at_idx, mol)

    # If terminal, don't make more specific
    if central_only and at_id in [0, 3]:  # Terminal atoms which we shouldn't specialise
        return f"[{ds['atomic_num']}{ds['degree']}:{at_id + 1}]"

    # The atom is central. Get all the bonded atoms and their bond types
    bonded_atoms = mol.GetAtomWithIdx(at_idx).GetNeighbors()
    bonded_atom_smirks = []
    for bonded_atom in bonded_atoms:
        bonded_idx = bonded_atom.GetIdx()
        bonded_ds = get_atom_descriptors(bonded_idx, mol)
        bond_type = get_bond_type((at_idx, bonded_idx), mol)
        bonded_atom_smirks.append(
            f"({bond_type}[{bonded_ds['atomic_num']}{bonded_ds['degree']}])"
        )

    return (
        f"[{ds['atomic_num']}{ds['degree']}:{at_id + 1}]{''.join(bonded_atom_smirks)}"
    )


get_atom_smirks_all_central_bonded = partial(
    _get_atom_smirks_all_bonded, central_only=True
)
get_atom_smirks_all_bonded = partial(_get_atom_smirks_all_bonded, central_only=False)


def get_atom_smirks_terminal_h_no_h(
    at_idx: int, at_id: int, mol: Chem.Mol, terminal_idxs: tuple[int, int]
) -> str:
    """A slightly more general SMIRKS representation for terminal atoms which replaces the
    atomic number with either a hydrogen or a non-hydrogen atom and ignore the degree.
    """
    ds = get_atom_descriptors(at_idx, mol)

    # If not terminal, remain specific
    if at_id not in terminal_idxs:  # Not terminal, remain specific
        return f"[{ds['atomic_num']}{ds['degree']}:{at_id + 1}]"

    # If terminal, check if hydrogen or not
    ds["atomic_num"] = ds["atomic_num"] if ds["atomic_num"] == "#1" else "!#1"

    return f"[{ds['atomic_num']}:{at_id + 1}]"


def get_atom_smirks_terminal_wildcard(
    at_idx: int, at_id: int, mol: Chem.Mol, terminal_indices: tuple[int, int]
) -> str:
    """A more general SMIRKS representation for terminal atoms which replaces the
    atomic number with a wildcard and ignore the degree and charge.
    """
    ds = get_atom_descriptors(at_idx, mol)

    # If not terminal, remain specific
    if at_id not in terminal_indices:  # Not terminal, remain specific
        return f"[{ds['atomic_num']}{ds['degree']}:{at_id + 1}]"

    # If terminal, use wildcard
    return f"[*:{at_id + 1}]"


def get_bond_smirks_standard(
    atom_idxs: tuple[int, int], central_bond: bool, mol: Chem.Mol
) -> str:
    """Generate the SMIRKS representation for a bond in the torsion.
    Standard specificity level.
    """
    return get_bond_type(atom_idxs, mol)


def get_bond_smirks_non_central_bonds_generalised(
    atom_idxs: tuple[int, int], central_bond: bool, mol: Chem.Mol
) -> str:
    """A slightly more general SMIRKS representation which only specifies the bond type for central bonds.
    """
    if not central_bond:
        return "~"
    else:
        return get_bond_type(atom_idxs, mol)


def get_bond_smirks_all_bonds_generalised(
    atom_idxs: tuple[int, int], central_bond: bool, mol: Chem.Mol
) -> str:
    """A more general SMIRKS representation which specifies the bond type for all bonds.
    """
    return "~"


def get_bond_idxs(mol: Molecule) -> set[tuple[int, int]]:
    """Get the indices of all bonds in a molecule.
    """
    return {
        tuple(sorted((a.molecule_atom_index for a in bond.atoms))) for bond in mol.bonds
    }


def get_angle_idxs(mol: Molecule) -> set[tuple[int, int, int]]:
    """Get the indices of all angles in a molecule.
    """
    return {tuple(a.molecule_atom_index for a in angle) for angle in mol.angles}


def get_proper_torsion_idxs(mol: Molecule) -> set[tuple[int, int, int, int]]:
    """Get the indices of all proper torsions in a molecule.
    """
    return {tuple(a.molecule_atom_index for a in torsion) for torsion in mol.propers}


def get_improper_torsion_idxs(mol: Molecule) -> set[tuple[int, int, int, int]]:
    """Get the indices of all improper torsions in a molecule.
    """
    return {tuple(a.molecule_atom_index for a in torsion) for torsion in mol.impropers}


class MMComponent(ABC):
    """A class to represent a molecular mechanics component (e.g., torsion, angle).
    """

    # __slots__ = ["mapped_smiles", "indices", "mol", "rdkit_mol", "n_atoms", "handler_class", "parameter_type", "getter_fn"]
    __slots__ = ["mapped_smiles", "indices", "mol", "rdkit_mol", "n_atoms"]

    mapped_smiles: str
    indices: tuple[int, ...]
    mol: Molecule
    rdkit_mol: Chem.Mol
    n_atoms: int  # Subclass must define number of atoms
    handler_class: type[ParameterHandler]  # Subclass must define handler class
    hander_version: float
    parameter_type: type[ParameterType]  # Subclass must define parameter type
    getter_fn: Callable[
        [Molecule], set[tuple[int, ...]]
    ]  # Subclass must define getter function

    def __init_subclass__(cls, **kwargs):
        """Enfore that certain class attributes are defined in subclasses."""
        super().__init_subclass__(**kwargs)

        required_attrs = [
            "n_atoms",
            "handler_class",
            "handler_version",
            "parameter_type",
            "getter_fn",
        ]

        for attr in required_attrs:
            if not hasattr(cls, attr):
                raise TypeError(f"{cls.__name__} must define class attribute '{attr}'")

    def __init__(self, indices: tuple[int, ...], mol: Molecule, rdkit_mol: Chem.Mol):
        self.indices = indices
        self.mol = mol
        self.rdkit_mol = rdkit_mol

        # Apply MDL aromaticity model
        Chem.SetAromaticity(self.rdkit_mol, Chem.AromaticityModel.AROMATICITY_MDL)

    @property
    def central_bond_index(self) -> Optional[int]:
        """Return the index of the central bond in the component, if applicable."""
        if self.n_atoms == 4:
            return 1

    @property
    def terminal_atom_indices(self) -> tuple[int, int]:
        """Return the indices of the terminal atoms in the component."""
        return (0, self.n_atoms - 1)

    def _construct_smirks(self, atoms: list[str], bonds: list[str]) -> str:
        """Construct the SMIRKS representation from atoms and bonds.
        """
        # Make sure that the lengths of atoms and bonds are consistent
        assert len(atoms) == self.n_atoms, (
            f"Expected {self.n_atoms} atoms, got {len(atoms)}"
        )
        assert len(bonds) == self.n_atoms - 1, (
            f"Expected {self.n_atoms - 1} bonds, got {len(bonds)}"
        )

        smirks = atoms[0]
        for bond, atom in zip(bonds, atoms[1:]):
            smirks += f"{bond}{atom}"

        return smirks

    def get_smirks(self, specificity_level: SpecificityLevel) -> str:
        """Generate the SMIRKS representation for the component.
        """
        idxs = self.indices
        n = self.n_atoms

        atoms_fwd = [
            specificity_level.get_atom_smirks(
                at_idx, at_id, self.rdkit_mol, self.terminal_atom_indices
            )
            for at_idx, at_id in zip(idxs, range(n))
        ]
        atoms_bwd = [
            specificity_level.get_atom_smirks(
                at_idx, at_id, self.rdkit_mol, self.terminal_atom_indices
            )
            for at_idx, at_id in zip(reversed(idxs), range(n))
        ]

        bonds_fwd = [
            specificity_level.get_bond_smirks(
                (idxs[j], idxs[j + 1]),
                central_bond=(j == self.central_bond_index),
                mol=self.rdkit_mol,
            )
            for j in range(n - 1)
        ]

        smirks_fwd = self._construct_smirks(atoms_fwd, bonds_fwd)
        smirks_bwd = self._construct_smirks(atoms_bwd, list(reversed(bonds_fwd)))

        # Return the lexicographically smallest SMIRKS representation to ensure
        # that the order of atoms does not affect the representation.
        return min(smirks_fwd, smirks_bwd)

    def matches_smirks(self, smirks: str) -> bool:
        """Check if the SMIRKS representation of the proper torsion matches the given SMIRKS.
        """
        indices_list = self.mol.chemical_environment_matches(smirks)

        if not indices_list:
            return False

        for indices in indices_list:
            if self.indices == indices:
                return True

        return False

    def __str__(self):
        return f"{self.mapped_smiles} ({self.indices})"

    @classmethod
    @abstractmethod
    def get_parameter(
        cls,
        smirks: str,
        specificity_num: int,
        components: list["MMComponent"],
        index: int,
        base_ff: ForceField,
    ) -> ParameterType:
        """Get the parameter for the component."""
        ...


def get_parameters_for_components(
    components: list[MMComponent], forcefield: ForceField, max_samples: int = 10
) -> ParameterList:
    """Get the parameters for a list of MM components from a force field. max_samples limits the number of components
    to get the parameters for, to speed this up.
    """
    parameters = []
    subsampled_components = (
        components
        if len(components) <= max_samples
        else np.random.choice(components, max_samples, replace=False)
    )
    for c in subsampled_components:
        assigned_parameters = forcefield.label_molecules(c.mol.to_topology())[
            0
        ]  # As only one molecule
        parameters.append(assigned_parameters[c.handler_class._TAGNAME][c.indices])
    return parameters


class Bond(MMComponent):
    """A class to represent a bond in a molecule."""

    n_atoms: int = 2
    handler_class: type[ParameterHandler] = BondHandler
    handler_version: float = 0.4
    parameter_type: type[ParameterType] = BondHandler.BondType
    getter_fn: Callable[[Molecule], set[tuple[int, ...]]] = get_bond_idxs

    @classmethod
    def get_parameter(
        cls,
        smirks: str,
        specificity_num: int,
        components: list["MMComponent"],
        index: int,
        base_ff: ForceField,
    ) -> BondHandler.BondType:
        # assert all(isinstance(c, Bond) for c in components), f"All components must be Bond instances but got {[type(c) for c in components]}"

        base_ff_parameters = get_parameters_for_components(components, base_ff)
        k_unit = off_unit.kilocalorie_per_mole / off_unit.angstroms**2
        mean_k = np.mean([p.k.m_as(k_unit) for p in base_ff_parameters]) * k_unit
        length_unit = off_unit.angstroms
        mean_length = (
            np.mean([p.length.m_as(length_unit) for p in base_ff_parameters])
            * length_unit
        )

        parameter = BondHandler.BondType(
            smirks=smirks,
            k=mean_k,
            length=mean_length,
            id=f"specificity={specificity_num} index={index} count={len(components)}",
        )
        return parameter


class Angle(MMComponent):
    """A class to represent an angle in a molecule."""

    n_atoms: int = 3
    handler_class: type[ParameterHandler] = AngleHandler
    handler_version: float = 0.3
    parameter_type: type[ParameterType] = AngleHandler.AngleType
    getter_fn: Callable[[Molecule], set[tuple[int, ...]]] = get_angle_idxs

    @classmethod
    def get_parameter(
        cls,
        smirks: str,
        specificity_num: int,
        components: list["MMComponent"],
        index: int,
        base_ff: ForceField,
    ) -> AngleHandler.AngleType:
        # assert all(isinstance(c, Angle) for c in components), f"All components must be Angle instances but got {[type(c) for c in components]}"

        base_ff_parameters = get_parameters_for_components(components, base_ff)
        k_unit = off_unit.kilocalorie_per_mole / off_unit.radians**2
        mean_k = np.mean([p.k.m_as(k_unit) for p in base_ff_parameters]) * k_unit
        angle_unit = off_unit.degrees
        mean_angle = (
            np.mean([p.angle.m_as(angle_unit) for p in base_ff_parameters]) * angle_unit
        )

        parameter = AngleHandler.AngleType(
            smirks=smirks,
            k=mean_k,
            angle=mean_angle,
            id=f"specificity={specificity_num} index={index} count={len(components)}",
        )
        return parameter


class ProperTorsion(MMComponent):
    """A class to represent a proper torsion in a molecule."""

    n_atoms: int = 4
    handler_class: type[ParameterHandler] = ProperTorsionHandler
    handler_version: float = 0.4
    parameter_type: type[ParameterType] = ProperTorsionHandler.ProperTorsionType
    getter_fn: Callable[[Molecule], set[tuple[int, ...]]] = get_proper_torsion_idxs

    @classmethod
    def get_parameter(
        cls,
        smirks: str,
        specificity_num: int,
        components: list["MMComponent"],
        index: int,
        base_ff: ForceField,
    ) -> ProperTorsionHandler.ProperTorsionType:
        # assert all(isinstance(c, ProperTorsion) for c in components), "All components must be ProperTorsion instances"
        parameter = ProperTorsionHandler.ProperTorsionType(
            smirks=smirks,
            k=[0 * off_unit.kilocalorie_per_mole / off_unit.radian**2]
            * 4,  # Default K values
            phase=[0 * _DEGREES] * 4,  # Default phase values
            periodicity=[1, 2, 3, 4],  # Default periodicities
            idivf=[1.0] * 4,  # Default idivf values
            id=f"specificity={specificity_num} index={index} count={len(components)}",
        )
        return parameter


class ImproperTorsion(MMComponent):
    """A class to represent an improper torsion in a molecule."""

    n_atoms: int = 4
    handler_class: type[ParameterHandler] = ImproperTorsionHandler
    handler_version: float = 0.3
    parameter_type: type[ParameterType] = ImproperTorsionHandler.ImproperTorsionType
    getter_fn: Callable[[Molecule], set[tuple[int, ...]]] = get_improper_torsion_idxs

    # Need to override some methods as impropers are not symmetrical in that
    # the third atom is always the central atom.
    def _construct_smirks(self, atoms: list[str], bonds: list[str]) -> str:
        """Construct the SMIRKS representation for the ImproperTorsion from atoms and bonds.
        """
        # Make sure that the lengths of atoms and bonds are consistent
        assert len(atoms) == self.n_atoms, (
            f"Expected {self.n_atoms} atoms, got {len(atoms)}"
        )
        assert len(bonds) == self.n_atoms - 1, (
            f"Expected {self.n_atoms - 1} bonds, got {len(bonds)}"
        )

        return (
            f"{atoms[0]}{bonds[0]}{atoms[1]}({bonds[1]}{atoms[2]}){bonds[2]}{atoms[3]}"
        )

    def get_smirks(self, specificity_level: SpecificityLevel) -> str:
        """Generate the SMIRKS representation for the ImproperTorsion.
        """
        idxs = self.indices
        n = self.n_atoms

        atoms = [
            specificity_level.get_atom_smirks(
                at_idx, at_id, self.rdkit_mol, self.terminal_atom_indices
            )
            for at_idx, at_id in zip(idxs, range(n))
        ]

        # Third atom is always central
        bond_atom_numbers = {(0, 1), (1, 2), (1, 3)}
        bonds = [
            specificity_level.get_bond_smirks(
                (idxs[i], idxs[j]), central_bond=(j == 2), mol=self.rdkit_mol
            )
            for i, j in bond_atom_numbers
        ]
        return self._construct_smirks(atoms, bonds)

    @classmethod
    def get_parameter(
        cls,
        smirks: str,
        specificity_num: int,
        components: list["MMComponent"],
        index: int,
        base_ff: ForceField,
    ) -> ImproperTorsionHandler.ImproperTorsionType:
        # assert all(isinstance(c, ImproperTorsion) for c in components), "All components must be ImproperTorsion instances"
        parameter = ImproperTorsionHandler.ImproperTorsionType(
            smirks=smirks,
            k=[
                0 * off_unit.kilocalorie_per_mole / off_unit.radian**2
            ],  # Default K value
            phase=[180 * _DEGREES],  # Default phase value
            periodicity=[2],  # Default periodicity
            idivf=[1.0],  # Default idivf value
            id=f"specificity={specificity_num} index={index} count={len(components)}",
        )
        return parameter


def get_mm_components_from_smiles(
    mapped_smiles: str, component_type: type[MMComponent]
) -> list[MMComponent]:
    """Get all MM components of a given type from a mapped SMILES string.
    Returns a list of MMComponent objects.
    """
    mol = Molecule.from_mapped_smiles(mapped_smiles, allow_undefined_stereo=True)
    if mol is None:
        raise ValueError(f"Invalid mapped SMILES: {mapped_smiles}")

    rdkit_mol = mol.to_rdkit()

    component_idxs = component_type.getter_fn(mol)
    return [
        component_type(
            indices=idxs,
            mol=mol,
            rdkit_mol=rdkit_mol,
        )
        for idxs in component_idxs
    ]


def get_all_mm_components(
    mapped_smiles_iterable: Iterable[str],
    component_type: type[MMComponent],
    unwanted_smirks: list[str] | None = None,
) -> list[MMComponent]:
    """Get all MM components of a given type from a set of mapped SMILES strings.

    Args:
        mapped_smiles_iterable (Iterable[str]): An iterable of mapped SMILES strings.
        unwanted_smirks (list[str] | None): A list of SMIRKS strings to filter out unwanted components. Defaults to None.

    Returns:
        list[MMComponent]: A list of MMComponent objects.

    """
    all_components = []
    for mapped_smiles in tqdm(mapped_smiles_iterable, desc="Processing Mapped SMILES"):
        try:
            all_components.extend(
                get_mm_components_from_smiles(mapped_smiles, component_type)
            )
        except ValueError as e:
            print(f"Skipping invalid mapped SMILES {mapped_smiles}: {e}")

    if unwanted_smirks:
        all_components_filtered = []
        print(f"Filtering out unwanted SMIRKS: {unwanted_smirks}")
        for component in tqdm(all_components, desc="Filtering unwanted components"):
            matched = False
            for smirks in unwanted_smirks:
                if component.matches_smirks(smirks):
                    matched = True
                    break
            if not matched:
                all_components_filtered.append(component)
        print(
            f"Filtered out {len(all_components) - len(all_components_filtered)} unwanted torsions."
        )
        all_components = all_components_filtered

    return all_components


def get_all_mm_components_by_type(
    mm_components: Iterable[MMComponent], specificity_level: SpecificityLevel
) -> dict[str, list[MMComponent]]:
    """Get the SMIRKS representations of all MM components in a set of molecules."""
    all_component_types = defaultdict(list)
    for component in tqdm(mm_components, desc="Processing Components"):
        smirks = component.get_smirks(specificity_level)
        all_component_types[smirks].append(component)

    return all_component_types


def process_mm_component_chunk(component_chunk, specificity_level):
    chunk_dict = defaultdict(list)
    for component in component_chunk:
        smirks = component.get_smirks(specificity_level)
        chunk_dict[smirks].append(component)
    return chunk_dict


def merge_dicts(dicts):
    merged = defaultdict(list)
    for d in dicts:
        for k, v in d.items():
            merged[k].extend(v)
    return merged


def get_all_mm_components_by_type_parallel(
    mm_components: Iterable[MMComponent],
    specificity_level: SpecificityLevel,
    n_workers=None,
) -> dict[str, list[MMComponent]]:
    mm_components = list(mm_components)
    if n_workers is None:
        n_workers = os.cpu_count()
    chunk_size = math.ceil(len(mm_components) / n_workers)
    component_chunks = [
        mm_components[i : i + chunk_size]
        for i in range(0, len(mm_components), chunk_size)
    ]

    with concurrent.futures.ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = [
            executor.submit(process_mm_component_chunk, chunk, specificity_level)
            for chunk in component_chunks
        ]
        results = []
        for f in tqdm(
            concurrent.futures.as_completed(futures),
            total=len(futures),
            desc="Processing Components (parallel)",
        ):
            results.append(f.result())

    return merge_dicts(results)


def get_mm_components_by_specificity_by_type(
    mm_components: Iterable[MMComponent],
    specificity_levels: dict[int, SpecificityLevel],
    cutoff_population: int = 10,
) -> dict[int, dict[str, list[MMComponent]]]:
    """Get the SMIRKS representations of all MM components in a set of molecules, grouped by specificity level.

    Args:
        mm_components (Iterable[MMComponent]): An iterable of MMComponent objects
        specificity_levels (dict[int, SpecificityLevel]): A dictionary mapping specificity level numbers to SpecificityLevel objects
        cutoff_population (int): The minimum number of occurrences for a component at a given specificity level before which the
        specificity is decreased to the next level. Defaults to 10.

    Returns:
        dict[int, dict[str, list[MMComponent]]]: A dictionary where keys are specificity level numbers and values are dictionaries mapping SMIRKS strings to lists of MMComponent objects.

    """
    components_by_specificity = {}
    specificity_order = sorted(specificity_levels.keys(), reverse=True)
    components_to_process = list(mm_components)

    for i, specificity_num in enumerate(specificity_order):
        specificity_level = specificity_levels[specificity_num]
        components_by_type = get_all_mm_components_by_type_parallel(
            components_to_process, specificity_level
        )
        components_by_specificity[specificity_num] = components_by_type

        # Prepare for next (less specific) level
        if i < len(specificity_order) - 1:
            # Find component types below cutoff
            too_specific = [
                component
                for component_list in components_by_type.values()
                if len(component_list) < cutoff_population
                for component in component_list
            ]
            # Remove these from current level
            for smirks in [
                s for s, cs in components_by_type.items() if len(cs) < cutoff_population
            ]:
                del components_by_specificity[specificity_num][smirks]
            components_to_process = too_specific

    return components_by_specificity


def get_mm_component_type_num(
    components_by_type: dict[str, list[MMComponent]],
) -> dict[str, int]:
    """Count the number of occurrences of each component type."""
    return {
        smirks: len(components) for smirks, components in components_by_type.items()
    }


def flatten_mm_component_types(
    components_by_specificity: dict[int, dict[str, list[MMComponent]]],
) -> dict[str, int]:
    """Flatten the component types dictionary to a single level with counts."""
    all_component_type_counts = Counter()
    for components_by_type in components_by_specificity.values():
        all_component_type_counts.update(get_mm_component_type_num(components_by_type))
    return dict(all_component_type_counts)


def summarise_all_types(
    mm_component_types: dict[int, dict[str, list[MMComponent]]],
) -> None:
    """Print a summary of the component types."""
    for specificity_num, components_by_type in mm_component_types.items():
        component_type_counts = get_mm_component_type_num(components_by_type)
        print(
            f"Total unique component types at specificity level {specificity_num}: {len(component_type_counts)}"
        )
        print("Most common component types:")
        for smirks, count in sorted(
            component_type_counts.items(), key=lambda item: item[1], reverse=True
        )[:10]:
            print(f"{smirks}: {count}")

    # Flatten the component types for overall statistics
    all_component_type_counts = flatten_mm_component_types(mm_component_types)
    print(
        f"% of component types with count < 5: {sum(1 for count in all_component_type_counts.values() if count < 5) / len(all_component_type_counts) * 100:.2f}%"
    )
    print(
        f"% of component types with count = 1: {sum(1 for count in all_component_type_counts.values() if count == 1) / len(all_component_type_counts) * 100:.2f}%"
    )

    # Print the total number of unique component types across all specificity levels
    total_unique_component_types = len(all_component_type_counts)
    print(
        f"Total unique component types across all specificity levels: {total_unique_component_types}"
    )


def plot_historgram_of_n_mol_per_type(
    mm_component_types: dict[int, dict[str, list[MMComponent]]],
) -> None:
    """Plot a histogram of the number of molecules per component type."""
    import matplotlib.pyplot as plt
    import seaborn as sns

    counts = flatten_mm_component_types(mm_component_types)
    fig, ax = plt.subplots(figsize=(10, 6))
    # Plot normalised histogram with KDE
    sns.histplot(counts, bins=30, kde=True, ax=ax, log_scale=True)  # , stat="density")
    ax.set_xlabel("Number of Molecules per Component Type")
    ax.set_ylabel("Frequency")
    ax.set_title("Histogram of Number of Molecules per Component Type")
    # Use log scale for better visibility
    plt.tight_layout()
    plt.show()


def plot_cdf_of_n_mol_per_type(
    mm_component_types: dict[int, dict[str, list[MMComponent]]],
) -> None:
    """Plot the CDF of the number of molecules per component type."""
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns

    counts = list(flatten_mm_component_types(mm_component_types).values())
    sorted_counts = np.log10(np.sort(counts))  # Use log scale for better visibility
    # sorted_counts = np.sort(counts)
    cdf = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.lineplot(x=sorted_counts, y=cdf, ax=ax)
    ax.set_xlabel("Log10(Number of Molecules per Component Type)")
    ax.set_ylabel("Cumulative Probability")
    ax.set_title("CDF of Number of Molecules per Component Type")
    plt.tight_layout()
    plt.show()


def _make_parameter(args):
    smirks, components, specificity_num, i, ff, component_class = args
    return component_class.get_parameter(smirks, specificity_num, components, i, ff)


def add_types_to_ff(
    ff: ForceField,
    component_types: dict[int, dict[str, list[MMComponent]]],
    component_class: type[MMComponent],
    extra_parameters: list[ParameterType] | None = None,
) -> ForceField:
    """Add component types to the force field."""
    ff_copy = deepcopy(ff)
    handler = component_class.handler_class(version=component_class.handler_version)

    # Write the lowest specificity level first
    for specificity_num, components_by_type in sorted(
        component_types.items(), key=lambda item: item[0]
    ):
        for i, (smirks, components) in tqdm(
            enumerate(
                sorted(
                    components_by_type.items(),
                    key=lambda item: len(item[1]),
                    reverse=True,
                )
            ),
            total=len(components_by_type),
            desc=f"Adding parameters for specificity {specificity_num}",
        ):
            parameter = component_class.get_parameter(
                smirks, specificity_num, components, i, ff
            )
            handler.parameters.append(parameter)

    # # Write the lowest specificity level first
    # for specificity_num, components_by_type in sorted(component_types.items(), key=lambda item: item[0]):
    #     # Prepare tasks in the same order as the serial version
    #     tasks = [
    #         (smirks, components, specificity_num, i, ff, component_class)
    #         for i, (smirks, components) in enumerate(
    #             sorted(
    #                 components_by_type.items(),
    #                 key=lambda item: len(item[1]),
    #                 reverse=True
    #             )
    #         )
    #     ]

    #     with ProcessPoolExecutor() as executor:
    #         # executor.map preserves ordering
    #         results = list(
    #             tqdm(
    #                 executor.map(_make_parameter, tasks),
    #                 total=len(tasks),
    #                 desc=f"Adding parameters for specificity {specificity_num}"
    #             )
    #         )

    #     # Append results in correct order
    #     for result in results:
    #         handler.parameters.append(result)

    # Add any extra parameters at the end
    if extra_parameters:
        for parameter in extra_parameters:
            handler.parameters.append(parameter)

    ff_copy.deregister_parameter_handler(component_class.handler_class._TAGNAME)
    ff_copy.register_parameter_handler(handler)

    return ff_copy


def check_molecule_can_be_parameterised(mapped_smiles: str, ff: ForceField) -> bool:
    """Check if a molecule can be fully parameterised by the force field. Accelerate this
    by skipping the charge assignment step (so that all errors are due to missing parameters).
    """
    mol = Molecule.from_mapped_smiles(mapped_smiles, allow_undefined_stereo=True)
    mol._partial_charges = [
        0.0 for _ in range(mol.n_atoms)
    ] * off_unit.elementary_charge  # Dummy charges to skip charge assignment
    try:
        Interchange.from_smirnoff(
            force_field=ff, topology=mol.to_topology()
        )  # , charge_from_molecules=True)
        return True
    except Exception as e:
        print(f"Error parameterising molecule {mapped_smiles}: {e}")
        return False


def check_molecules_fully_covered_chunk(smiles_chunk, ff):
    return [
        (smiles, check_molecule_can_be_parameterised(smiles, ff))
        for smiles in smiles_chunk
    ]


def chunked_iterable(iterable, chunk_size):
    """Yield successive chunk_size-sized chunks from iterable."""
    it = iter(iterable)
    while True:
        chunk = []
        try:
            for _ in range(chunk_size):
                chunk.append(next(it))
        except StopIteration:
            if chunk:
                yield chunk
            break
        if chunk:
            yield chunk


def check_all_molecules_parameterisable_parallel_chunks(
    mapped_smiles_list, ff, n_workers=None
):
    results = []
    if n_workers is None:
        import os

        n_workers = os.cpu_count()
    # Calculate the chunk size to be the maximum possible while still having at least one chunk per worker
    chunk_size = math.ceil(len(mapped_smiles_list) / n_workers)
    chunks = list(chunked_iterable(mapped_smiles_list, chunk_size))
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = [
            executor.submit(check_molecules_fully_covered_chunk, chunk, ff)
            for chunk in chunks
        ]
        for f in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Checking Parameterisability (parallel chunks)",
        ):
            results.extend(f.result())
    return results


def check_all_components_fully_covered(
    mapped_smiles: list[str], ff: ForceField
) -> dict[str, dict[str, list[tuple[int, ...]]]]:
    """Check if all components (bonds, angles, proper torsions, improper torsions) in a molecule are covered by the force field.

    Args:
        mapped_smiles (str): The mapped SMILES string of the molecule.
        ff (ForceField): The force field to check against.

    Returns:
        dict | None: A of components types with missing parameters, keyed by smiles string. Each value is a
        dict with keys being component types and values being lists of tuples of atom indices with missing parameters.

    """
    all_unassigned = {}
    for smiles in tqdm(mapped_smiles, desc="Checking Component Coverage"):
        mol = Molecule.from_mapped_smiles(smiles, allow_undefined_stereo=True)
        top = Topology.from_molecules([mol])
        labels = ff.label_molecules(top)[0]  # As only one molecule

        unassigned = {}
        for component_class in [Bond, Angle, ProperTorsion, ImproperTorsion]:
            component_type = component_class.handler_class._TAGNAME
            component_params = labels[
                component_type
            ]  # dict with keys being atom index tuples and values being parameters
            covered_indices = list(component_params.keys())
            unassigned_indices = []
            for at_indices in component_class.getter_fn(mol):
                if at_indices not in covered_indices:
                    unassigned_indices.append(at_indices)
            if unassigned_indices:
                unassigned[component_type] = unassigned_indices
                print(covered_indices)
                print(component_class.getter_fn(mol))

        if unassigned:
            all_unassigned[smiles] = unassigned

    return all_unassigned


def check_all_components_fully_covered_parallel_chunks(
    mapped_smiles_list: list[str], ff: ForceField, n_workers=None
) -> dict[str, dict[str, list[tuple[int, ...]]]]:
    results = {}
    if n_workers is None:
        import os

        n_workers = os.cpu_count()
    # Calculate the chunk size to be the maximum possible while still having at least one chunk per worker
    chunk_size = math.ceil(len(mapped_smiles_list) / n_workers)
    chunks = list(chunked_iterable(mapped_smiles_list, chunk_size))
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = [
            executor.submit(check_all_components_fully_covered, chunk, ff)
            for chunk in chunks
        ]
        for f in tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Checking Component Coverage (parallel chunks)",
        ):
            results.update(f.result())
    return results


def check_torsions_fully_covered(
    mapped_smiles: str, ff: ForceField
) -> tuple[bool, list[tuple[int, int, int, int]]]:
    """Check if all proper torsions in a molecule are covered by the force field.

    Args:
        mapped_smiles (str): The mapped SMILES string of the molecule.
        ff (ForceField): The force field to check against.

    Returns:
        tuple: A tuple containing a boolean indicating if all torsions are covered and a list of unassigned torsion indices.

    """
    mol = Molecule.from_mapped_smiles(mapped_smiles, allow_undefined_stereo=True)
    # mol.generate_conformers(n_conformers=1)  # Needed to avoid issues in Topology

    top = Topology.from_molecules([mol])
    labels = ff.label_molecules(top)
    torsion_params = labels[0][
        "ProperTorsions"
    ]  # list of (parameter, atom_indices) tuples
    covered_indices = list(torsion_params.keys())

    unassigned = []
    for torsion in mol.propers:
        atom_indices = tuple(a.molecule_atom_index for a in torsion)
        if atom_indices not in covered_indices:
            unassigned.append(atom_indices)

    return len(unassigned) == 0, unassigned


def show_molecule_with_atom_indices(
    mapped_smiles: str, highlight_indices: tuple[int, ...]
) -> Draw.MolToImage:
    # Parse the molecule from the mapped SMILES
    mol = Molecule.from_mapped_smiles(mapped_smiles, allow_undefined_stereo=True)
    rdmol = mol.to_rdkit()  # Convert to RDKit molecule for visualization

    rdDepictor.Compute2DCoords(rdmol)

    # Set up drawer
    drawer = rdMolDraw2D.MolDraw2DCairo(300, 300)
    options = drawer.drawOptions()

    # Add red labels only for specified atoms
    for idx in highlight_indices:
        options.atomLabels[idx] = f"{idx}"

    # Draw with highlights
    drawer.DrawMolecule(
        rdmol,
        highlightAtoms=highlight_indices,
        # Colour the highlighted atoms green
        highlightAtomColors={idx: (0.0, 1.0, 0.0) for idx in highlight_indices},
    )
    drawer.FinishDrawing()

    # Convert to PIL image
    img_data = drawer.GetDrawingText()
    return Image.open(io.BytesIO(img_data))

In [3]:
SPECIFICITY_LEVELS_BY_COMPONENT: dict[
    type[MMComponent], dict[int, SpecificityLevel]
] = {
    Bond: {
        0: SpecificityLevel(
            name="Standard",
            get_atom_smirks=get_atom_smirks_standard,
            get_bond_smirks=get_bond_smirks_standard,
        ),
    },
    Angle: {
        0: SpecificityLevel(
            name="TerminalWildcard",
            get_atom_smirks=get_atom_smirks_terminal_wildcard,
            get_bond_smirks=get_bond_smirks_non_central_bonds_generalised,
        ),
        1: SpecificityLevel(
            name="TerminalHnoH",
            get_atom_smirks=get_atom_smirks_terminal_h_no_h,
            get_bond_smirks=get_bond_smirks_non_central_bonds_generalised,
        ),
        2: SpecificityLevel(
            name="Standard",
            get_atom_smirks=get_atom_smirks_standard,
            get_bond_smirks=get_bond_smirks_standard,
        ),
    },
    ProperTorsion: {
        0: SpecificityLevel(
            name="TerminalWildcard",
            get_atom_smirks=get_atom_smirks_terminal_wildcard,
            get_bond_smirks=get_bond_smirks_non_central_bonds_generalised,
        ),
        1: SpecificityLevel(
            name="TerminalHnoH",
            get_atom_smirks=get_atom_smirks_terminal_h_no_h,
            get_bond_smirks=get_bond_smirks_non_central_bonds_generalised,
        ),
        2: SpecificityLevel(
            name="Standard",
            get_atom_smirks=get_atom_smirks_standard,
            get_bond_smirks=get_bond_smirks_standard,
        ),
    },
    ImproperTorsion: {
        0: SpecificityLevel(
            name="TerminalWildcard",
            get_atom_smirks=get_atom_smirks_terminal_wildcard,
            get_bond_smirks=get_bond_smirks_non_central_bonds_generalised,
        ),
        1: SpecificityLevel(
            name="TerminalHnoH",
            get_atom_smirks=get_atom_smirks_terminal_h_no_h,
            get_bond_smirks=get_bond_smirks_non_central_bonds_generalised,
        ),
        2: SpecificityLevel(
            name="Standard",
            get_atom_smirks=get_atom_smirks_standard,
            get_bond_smirks=get_bond_smirks_standard,
        ),
    },
}

UNWANTED_SMIRKS_BY_COMPONENT: dict[type[MMComponent], list[str] | None] = {
    Bond: None,
    Angle: None,
    ProperTorsion: linear_torsion_smirks,
    ImproperTorsion: None,
}

EXTRA_PARAMETERS_BY_COMPONENT: dict[type[MMComponent], list[ParameterType] | None] = {
    Bond: None,
    Angle: None,
    ProperTorsion: linear_torsion_parameters,
    ImproperTorsion: None,
}

components_by_type: dict[type[MMComponent], list[MMComponent]] = {}

for component_class in [Bond, Angle, ProperTorsion, ImproperTorsion]:
    print(f"\n{'=' * 20}\nProcessing {component_class.__name__}\n{'=' * 20}")
    components = get_all_mm_components(
        SMILES_TRAIN_FILTERED,
        component_class,
        unwanted_smirks=UNWANTED_SMIRKS_BY_COMPONENT[component_class],
    )
    print(f"Found {len(components)} {component_class.__name__}s.")
    class_components_by_type = get_mm_components_by_specificity_by_type(
        components,
        SPECIFICITY_LEVELS_BY_COMPONENT[component_class],
        cutoff_population=10,
    )
    summarise_all_types(class_components_by_type)
    components_by_type[component_class] = class_components_by_type


Processing Bond


Processing Mapped SMILES: 100%|██████████| 22907/22907 [00:59<00:00, 382.54it/s]

Found 909591 Bonds.



Processing Components (parallel): 100%|██████████| 32/32 [00:40<00:00,  1.28s/it]


Total unique component types at specificity level 0: 149
Most common component types:
[#1X1:1]-[#6X4:2]: 230252
[#6X3:1]:[#6X3:2]: 142724
[#1X1:1]-[#6X3:2]: 111336
[#6X4:1]-[#6X4:2]: 67217
[#6X3:1]-[#7X3:2]: 45066
[#6X3:1]-[#6X4:2]: 39051
[#6X4:1]-[#7X3:2]: 29839
[#6X3:1]-[#6X3:2]: 29680
[#1X1:1]-[#7X3:2]: 24385
[#6X3:1]=[#8X1:2]: 21538
% of component types with count < 5: 18.12%
% of component types with count = 1: 10.07%
Total unique component types across all specificity levels: 149

Processing Angle


...

Total unique component types at specificity level 2: 1720
Most common component types:
[#1X1:1]-[#6X4:2]-[#6X4:3]-[#6X4:4]: 188171
[#1X1:1]-[#6X4:2]-[#6X4:3]-[#1X1:4]: 167377
[#1X1:1]-[#6X3:2]:[#6X3:3]:[#6X3:4]: 164061
[#6X3:1]:[#6X3:2]:[#6X3:3]:[#6X3:4]: 138442
[#1X1:1]-[#6X3:2]:[#6X3:3]-[#1X1:4]: 47495
[#1X1:1]-[#6X4:2]-[#6X4:3]-[#6X3:4]: 45978
[#1X1:1]-[#6X4:2]-[#6X4:3]-[#7X3:4]: 44405
[#1X1:1]-[#6X4:2]-[#7X3:3]-[#6X4:4]: 43392
[#1X1:1]-[#6X4:2]-[#7X3:3]-[#6X3:4]: 42769
[#6X4:1]-[#6X4:2]-[#6X4:3]-[#6X4:4]: 38088
Total unique component types at specificity level 1: 100
Most common component types:
[!#1:1]~[#6X3:2]-[#7X3:3]~[!#1:4]: 360
[!#1:1]~[#6X3:2]-[#6X3:3]~[!#1:4]: 286
[!#1:1]~[#6X3:2]-[#6X4:3]~[!#1:4]: 266
[!#1:1]~[#6X3:2]:[#6X3:3]~[!#1:4]: 235
[!#1:1]~[#6X4:2]-[#7X3:3]~[!#1:4]: 222
[!#1:1]~[#6X3:2]-[#7X2:3]~[!#1:4]: 205
[!#1:1]~[#6X3:2]-[#8X2:3]~[!#1:4]: 178
[!#1:1]~[#16X2:2]-[#6X3:3]~[!#1:4]: 176
[!#1:1]~[#6X3:2]=[#6X3:3]~[!#1:4]: 167
[!#1:1]~[#15X4:2]-[#6X3:3]~[!#1:4]: 160
T

Processing Mapped SMILES: 100%|██████████| 22907/22907 [01:42<00:00, 224.39it/s]


Found 4475910 ImproperTorsions.


Processing Components (parallel): 100%|██████████| 31/31 [00:00<00:00, 129.27it/s]


Total unique component types at specificity level 2: 2636
Most common component types:
[#1X1:1]-[#6X4:2](-[#6X4:3])-[#1X1:4]: 227900
[#1X1:1]-[#6X4:2](-[#1X1:3])-[#6X4:4]: 227900
[#6X4:1]-[#6X4:2](-[#1X1:3])-[#1X1:4]: 227900
[#1X1:1]-[#6X4:2](-[#1X1:3])-[#1X1:4]: 188700
[#6X3:1]:[#6X3:2](:[#6X3:3])-[#1X1:4]: 161862
[#1X1:1]-[#6X3:2](:[#6X3:3]):[#6X3:4]: 161862
[#6X3:1]:[#6X3:2](-[#1X1:3]):[#6X3:4]: 161862
[#6X4:1]-[#6X4:2](-[#6X4:3])-[#1X1:4]: 122338
[#1X1:1]-[#6X4:2](-[#6X4:3])-[#6X4:4]: 122338
[#6X4:1]-[#6X4:2](-[#1X1:3])-[#6X4:4]: 122338
Total unique component types at specificity level 1: 162
Most common component types:
[!#1:1]~[#6X4:2](-[#1X1:3])~[!#1:4]: 252
[!#1:1]~[#6X4:2](-[#6X3:3])~[!#1:4]: 250
[!#1:1]~[#7X3:2](-[#6X4:3])~[!#1:4]: 246
[!#1:1]~[#7X3:2](-[#6X3:3])~[!#1:4]: 242
[!#1:1]~[#6X3:2](=[#6X3:3])~[!#1:4]: 212
[!#1:1]~[#6X4:2](-[#6X4:3])~[!#1:4]: 210
[!#1:1]~[#6X3:2](=[#7X2:3])~[!#1:4]: 202
[!#1:1]~[#6X3:2](-[#7X3:3])~[!#1:4]: 200
[!#1:1]~[#15X4:2](-[#8X2:3])~[!#1:4]: 1

In [ ]:
# Write out to file
new_ff = ForceField("../input_ff/lj-sage-2-2-msm-0-expanded-torsions.offxml")
for component_class, angles_by_type in components_by_type.items():
    print(f"\nAdding {component_class.__name__} parameters to force field...")
    new_ff = add_types_to_ff(
        new_ff,
        angles_by_type,
        component_class,
        EXTRA_PARAMETERS_BY_COMPONENT[component_class],
    )

new_ff.to_file("generated_ff.offxml")


Adding Bond parameters to force field...


Adding parameters for specificity 0: 100%|██████████| 149/149 [00:59<00:00,  2.49it/s]



Adding Angle parameters to force field...


Adding parameters for specificity 2: 100%|██████████| 484/484 [02:05<00:00,  3.87it/s]



Adding ProperTorsion parameters to force field...


Adding parameters for specificity 2: 100%|██████████| 1720/1720 [00:00<00:00, 6421.00it/s]



Adding ImproperTorsion parameters to force field...


Adding parameters for specificity 2: 100%|██████████| 2636/2636 [00:00<00:00, 7423.79it/s]


In [3]:
new_ff = ForceField("generated_ff.offxml")

In [ ]:
COVERAGE_DATASETS = {
    "Train": SMILES_TRAIN_FILTERED,  # Sanity check
    "Test": smiles_data["test"],
    "Biaryl": BIARYL_SMILES,
    "TorsionNet": TNET_SMILES,
    "Industry Benchmark": INDUSTRY_BENCHMARK_SMILES,
}

for dataset_name, smiles in COVERAGE_DATASETS.items():
    print(f"\nChecking coverage for {dataset_name} dataset...")
    # uncovered = check_all_components_fully_covered_parallel_chunks(smiles, new_ff)
    uncovered = check_all_components_fully_covered_parallel_chunks(smiles, new_ff)
    if uncovered:
        total_uncovered = sum(len(v) for v in uncovered.values())
        print([v.keys() for v in uncovered.values()])
        print(uncovered)
        print(
            f"Found {total_uncovered} uncovered components in {len(uncovered)} component types:"
        )